<img src="logo.png" alt="Vegeta" width="240">

# AI design copilot — iterate on a CadQuery design with Claude

`vegeta.ai` lets Claude **propose** changes to a Dedalus design file. Vegeta **builds and measures**
every proposal in a temporary copy and shows the diff; the file changes only when **you accept**.

Set `ANTHROPIC_API_KEY` to use Claude. Without a key this notebook runs the same workflow with a small
scripted stand-in proposer, so you can see every step.

In [ ]:
import os, shutil
from pathlib import Path
from vegeta import dedalus
from vegeta.ai import ClaudeConfig, ClaudeProposer, DesignSession
from vegeta.dedalus import viz

RUNS = Path("_runs/ai"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
design_file = RUNS / "bracket.py"
design_file.write_text('''
import cadquery as cq
from vegeta.dedalus import Design, Parameter

class Bracket(Design):
    parameters = [
        Parameter("length", 80.0, "mm", min=20),
        Parameter("width", 40.0, "mm", min=10),
        Parameter("thickness", 6.0, "mm", min=1),
        Parameter("hole_diameter", 6.5, "mm", min=1),
    ]

    def build(self, p):
        plate = cq.Workplane("XY").box(p["length"], p["width"], p["thickness"])
        dx, dy = p["length"] / 2 - 10, p["width"] / 2 - 10
        return plate.faces(">Z").workplane().pushPoints([(dx, dy), (-dx, dy), (dx, -dy), (-dx, -dy)]).hole(p["hole_diameter"])
''')
print(design_file.read_text())

## 1. Choose the proposer
Claude when a key is available — every knob is an argument (`api_key`, `model`, `effort`, `max_tokens`, `extra_system`).

In [ ]:
HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"))

class DemoProposer:
    """Scripted stand-in used when no API key is set: returns pre-written proposals in order."""
    name = "demo"
    def __init__(self):
        self.turn = 0
    def describe(self):
        return {"provider": "demo", "model": "scripted"}
    def propose(self, context, instruction, history):
        self.turn += 1
        src = context["source"]
        if self.turn == 1:   # add two stiffening ribs along the length
            new = src.replace(
                '        return plate.faces(">Z")',
                '        rib = cq.Workplane("XY").box(p["length"] - 20, p["rib_width"], p["rib_height"]).translate((0, 0, p["thickness"] / 2 + p["rib_height"] / 2))\n'
                '        plate = plate.union(rib.translate((0, p["width"] / 4, 0))).union(rib.translate((0, -p["width"] / 4, 0)))\n'
                '        return plate.faces(">Z")').replace(
                '        Parameter("hole_diameter", 6.5, "mm", min=1),',
                '        Parameter("hole_diameter", 6.5, "mm", min=1),\n        Parameter("rib_width", 3.0, "mm", min=1),\n        Parameter("rib_height", 6.0, "mm", min=1),')
            return ({"kind": "source", "summary": "add two longitudinal ribs on top of the plate",
                     "rationale": "ribs raise the second moment of area far more than extra plate thickness for the same mass",
                     "source": new, "parameters": {}, "expected_effects": ["volume up by a few percent", "bending stiffness up"],
                     "risks": ["ribs cross the hole pattern if the holes move inboard"]},
                    {"input_tokens": 0, "output_tokens": 0, "model": "scripted"})
        return ({"kind": "parameters", "summary": "thinner plate now that the ribs carry bending",
                 "rationale": "the ribs dominate stiffness; the plate can lose 2 mm", "source": None,
                 "parameters": {"thickness": 4.0}, "expected_effects": ["volume down"], "risks": ["check the hole edges"]},
                {"input_tokens": 0, "output_tokens": 0, "model": "scripted"})

proposer = ClaudeProposer(ClaudeConfig(model="claude-opus-5", effort="high")) if HAVE_KEY else DemoProposer()
session = DesignSession(f"{design_file}:Bracket", proposer, parameters={"thickness": 6.0},
                        notes="clamped along one short edge, 200 N downward at the other; FDM-printed PETG")
print("using", proposer.describe())

## 2. Ask for a change — the file is untouched until you accept

In [ ]:
p1 = session.ask("add two stiffening ribs along the length; keep the hole pattern; add parameters for the rib size")
p1

In [ ]:
viz.show(viz.plot3d(session.geometry(p1)))     # look at the proposal before deciding

In [ ]:
print("file changed?", design_file.read_text() != p1.source_before)
session.accept(p1, note="ribs are what we wanted")
print("accepted -> backups:", sorted(f.name for f in RUNS.glob("bracket.py.*.bak")))

## 3. Keep iterating — the conversation continues with the accepted design as context

In [ ]:
p2 = session.ask("with the ribs in place, can the plate be thinner? propose values only")
p2

In [ ]:
if p2.ok and p2.kind != "answer":
    session.accept(p2)
before, after = p1.validation.measurements_before["volume"], session.geometry(p2).measure()["volume"]
print(f"volume: {before:.0f} -> {after:.0f} mm^3")

## 4. Everything is logged
Proposals, validations, token usage and your decisions, next to the design file.

In [ ]:
import json
for line in session.log.read_text().splitlines():
    r = json.loads(line)
    print(r["event"], r["proposal"]["id"], r["proposal"]["kind"], "-", r["proposal"]["summary"], "|", r["proposal"]["status"])

## 5. The accepted design is an ordinary Dedalus design
Generate it, export it, or register it in a `vegeta.core` workspace as a new revision.

In [ ]:
from vegeta.dedalus.loading import load_design
final = load_design(f"{design_file}:Bracket").generate(**session.parameters)
final.measure()